# El Mundo

In [ ]:
import requests
from bs4 import BeautifulSoup
import re


# settings

url = (
    "https://www.elmundo.es/elmundo/hemeroteca/2022/02/24/t/index.html"
)


# headers

headers = {

    "User-Agent": (

        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"

    )

}


# fetch

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

html = response.text


print("\nstatus\n")

print(
    response.status_code
)


print("\nhtml size\n")

print(
    len(html)
)


# soup

soup = BeautifulSoup(
    html,
    "html.parser"
)


# all linksп

links = soup.find_all(
    "a",
    href=True
)


all_urls = []


for link in links:

    href = link["href"]

    all_urls.append(
        href
    )


# deduplicate

all_urls = list(
    set(all_urls)
)


print("\ntotal unique links\n")

print(
    len(all_urls)
)


# article detection

article_urls = []


for url in all_urls:

    lower = url.lower()


    if re.search(

        r"/20\d{2}/\d{2}/\d{2}/",

        lower

    ):

        article_urls.append(
            url
        )


print("\narticle urls found\n")

print(
    len(article_urls)
)


# sample urls

print("\nsample article urls\n")

for u in article_urls[:100]:

    print(u)


# image check

images = soup.find_all("img")


print("\nimages found\n")

print(
    len(images)
)


# sample image urls

print("\nsample image urls\n")

for img in images[:20]:

    src = img.get("src")

    if src:

        print(src)


print("\nfinished")

In [ ]:
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import pandas as pd
import re
import time
from datetime import datetime
from datetime import timedelta


# settings

start_date = "2022-01-01"
end_date = "2022-01-31"

max_days = 10

sample_per_section = 5


# headers

headers = {

    "User-Agent": (

        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"

    )

}


session = requests.Session()

session.headers.update(headers)


# generate dates

start_dt = datetime.strptime(
    start_date,
    "%Y-%m-%d"
)

end_dt = datetime.strptime(
    end_date,
    "%Y-%m-%d"
)

all_dates = []

current = start_dt

while current <= end_dt:

    all_dates.append(current)

    current += timedelta(days=1)


# optional limit

all_dates = all_dates[:max_days]


print("\narchive days\n")

print(
    len(all_dates)
)


# build archive urls

archive_urls = []


for dt in all_dates:

    archive_url = (

        "https://www.elmundo.es/"
        "elmundo/hemeroteca/"
        f"{dt.year}/"
        f"{dt.month:02d}/"
        f"{dt.day:02d}/"
        "t/index.html"

    )


    archive_urls.append({

        "date": dt.strftime("%Y-%m-%d"),

        "year": dt.year,

        "archive_url": archive_url

    })


# fetch helper

def fetch_html(url):

    response = session.get(
        url,
        timeout=30
    )

    response.raise_for_status()

    return response.text


# collect articles

all_articles = []

seen_urls = set()


for row in tqdm(archive_urls):

    archive_url = row["archive_url"]

    archive_date = row["date"]

    archive_year = row["year"]


    try:

        html = fetch_html(
            archive_url
        )

        soup = BeautifulSoup(
            html,
            "html.parser"
        )


        links = soup.find_all(
            "a",
            href=True
        )


        for link in links:

            url = link["href"]


            if not isinstance(url, str):
                continue


            url = url.strip()


            # only elmundo

            if not url.startswith(
                "https://www.elmundo.es/"
            ):
                continue


            # remove anchors

            url = url.split("#")[0]


            # remove params

            url = url.split("?")[0]


            # article pattern

            if not re.search(

                r"/20\d{2}/\d{2}/\d{2}/",

                url

            ):
                continue


            # must end with html

            if not url.endswith(".html"):
                continue


            # remove duplicates

            if url in seen_urls:
                continue


            seen_urls.add(url)


            # path parts

            path_parts = [

                p
                for p in url
                .replace(
                    "https://www.elmundo.es/",
                    ""
                )
                .split("/")
                if p

            ]


            if len(path_parts) < 4:
                continue


            section = path_parts[0]


            all_articles.append({

                "archive_date": archive_date,

                "archive_year": archive_year,

                "section": section,

                "article_url": url

            })


        time.sleep(0.3)


    except Exception as e:

        print("\narchive error")

        print(archive_url)

        print(e)


# dataframe

articles_df = pd.DataFrame(
    all_articles
)


# diagnostics

print("\narticle urls\n")

print(
    len(articles_df)
)


print("\nunique sections\n")

print(
    articles_df["section"]
    .nunique()
)


print("\nsection distribution\n")

section_counts = (

    articles_df["section"]
    .value_counts()

)

print(
    section_counts
)


# sample urls per section

print("\nsample urls by section\n")


for section in section_counts.index:

    print("\n" + "=" * 80)

    print(section)

    print("=" * 80)


    sample_urls = (

        articles_df[
            articles_df["section"] == section
        ]["article_url"]

        .head(sample_per_section)

    )


    for url in sample_urls:

        print(url)


# save diagnostics

articles_df.to_csv(

    "elmundo_section_diagnostics.csv",

    index=False,

    encoding="utf-8-sig"

)


print("\ndiagnostics saved")

print(
    "elmundo_section_diagnostics.csv"
)

print("\nfinished")

In [ ]:
import re
import csv
import asyncio
import aiohttp

from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from datetime import datetime
from datetime import timedelta


# settings

headers = {
    "User-Agent": "Mozilla/5.0"
}

start_date = "2022-01-01"
end_date = "2026-12-31"

concurrency = 50

output_csv = "elmundo_corpus.csv"


# target sections

allowed_sections = {

    "internacional",
    "espana",
    "opinion",
    "economia",
    "sociedad",
    "cronica",
    "papel",
    "madrid",
    "cataluna",
    "comunidad-valenciana",
    "andalucia",
    "pais-vasco",
    "baleares"

}


# csv

csv_file = open(
    output_csv,
    "w",
    newline="",
    encoding="utf-8-sig"
)

writer = csv.DictWriter(
    csv_file,
    fieldnames=[
        "archive_date",
        "section",
        "url",
        "title",
        "published_time",
        "text",
        "image_count",
        "image_urls",
    ]
)

writer.writeheader()


# dates

start_dt = datetime.strptime(
    start_date,
    "%Y-%m-%d"
)

end_dt = datetime.strptime(
    end_date,
    "%Y-%m-%d"
)

dates = []

current = start_dt

while current <= end_dt:

    dates.append(current)

    current += timedelta(days=1)


print("\narchive days\n")

print(len(dates))


# helpers

def build_archive_url(dt):

    return (

        "https://www.elmundo.es/"
        "elmundo/hemeroteca/"
        f"{dt.year}/"
        f"{dt.month:02d}/"
        f"{dt.day:02d}/"
        "t/index.html"

    )


def extract_section(url):

    try:

        path = url.replace(
            "https://www.elmundo.es/",
            ""
        )

        return path.split("/")[0]

    except:

        return None


def clean_url(url):

    if not url:
        return None


    if not isinstance(url, str):
        return None


    lower = url.lower()


    # must look like image

    if not any(

        ext in lower

        for ext in [

            ".jpg",
            ".jpeg",
            ".png",
            ".webp",
            ".avif"

        ]

    ):

        return None


    # remove obvious junk

    bad_terms = [

        "logo",
        "icon",
        "avatar",
        "tracking",
        "pixel",
        "sprite",
        "favicon",
        "ads",
        "banner",
        "button"

    ]


    if any(

        bad in lower

        for bad in bad_terms

    ):

        return None


    # allow relative urls

    if lower.startswith("//"):

        url = "https:" + url


    elif lower.startswith("/"):

        url = "https://www.spiegel.de" + url


    return url


def is_article_url(url):

    pattern = r"/20\d{2}/\d{2}/\d{2}/"

    if not re.search(pattern, url):
        return False

    if not url.endswith(".html"):
        return False

    return True


def extract_image_urls(soup):

    images = set()


    for img in soup.find_all("img"):

        src = img.get("src")

        if not src:
            continue

        if not src.startswith("http"):
            continue


        low = src.lower()


        bad = [

            ".svg",
            "logo",
            "icon",
            "avatar",
            "pixel",
            "ads",
            "banner"

        ]


        if any(x in low for x in bad):
            continue


        images.add(src)


    og = soup.find(
        "meta",
        property="og:image"
    )

    if og:

        content = og.get("content")

        if content:

            images.add(content)


    return list(images)


# fetch

async def fetch_html(
    session,
    url
):

    try:

        async with session.get(
            url,
            timeout=20
        ) as r:

            if r.status != 200:
                return None

            return await r.text()

    except:

        return None


# archive parser

async def extract_article_urls(
    session,
    archive_url
):

    html = await fetch_html(
        session,
        archive_url
    )

    if not html:
        return []


    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    found = set()


    for a in soup.find_all(
        "a",
        href=True
    ):

        url = a["href"]


        if not isinstance(url, str):
            continue


        if not url.startswith(
            "https://www.elmundo.es/"
        ):
            continue


        url = clean_url(url)


        if not is_article_url(url):
            continue


        section = extract_section(url)

        if section not in allowed_sections:
            continue


        found.add(url)


    return list(found)


# metadata extraction

async def extract_article_data(
    session,
    url,
    archive_date
):

    try:

        html = await fetch_html(
            session,
            url
        )

        if not html:
            return None


        soup = BeautifulSoup(
            html,
            "html.parser"
        )


        # title

        title = None

        h1 = soup.find("h1")

        if h1:

            title = h1.get_text(
                strip=True
            )


        # publication time

        published_time = None

        meta_time = soup.find(

            "meta",

            property="article:published_time"

        )

        if meta_time:

            published_time = meta_time.get(
                "content"
            )


        # text

        paragraphs = []


        for p in soup.find_all("p"):

            text = p.get_text(
                " ",
                strip=True
            )


            if len(text) < 40:
                continue


            paragraphs.append(text)


        article_text = "\n".join(
            paragraphs
        )


        # images

        image_urls = extract_image_urls(
            soup
        )


        return {

            "archive_date": archive_date,

            "section": extract_section(url),

            "url": url,

            "title": title,

            "published_time": published_time,

            "text": article_text,

            "image_count": len(image_urls),

            "image_urls": " | ".join(
                image_urls
            )

        }

    except Exception as e:

        print("\narticle error")

        print(url)

        print(e)

        return None


# worker

async def process_article(

    session,
    semaphore,
    url,
    archive_date,
    pbar

):

    async with semaphore:

        data = await extract_article_data(

            session,
            url,
            archive_date

        )

        if data:

            writer.writerow(data)

            pbar.update(1)


# main

async def main():

    connector = aiohttp.TCPConnector(
        limit=concurrency
    )

    timeout = aiohttp.ClientTimeout(
        total=30
    )

    semaphore = asyncio.Semaphore(
        concurrency
    )

    pbar = tqdm(
        desc="articles",
        unit="article"
    )

    async with aiohttp.ClientSession(

        headers=headers,
        connector=connector,
        timeout=timeout

    ) as session:


        # step 1
        # collect article urls

        print("\ncollecting urls\n")


        all_urls = []


        for dt in tqdm(dates):

            archive_url = build_archive_url(dt)

            urls = await extract_article_urls(

                session,
                archive_url

            )


            for url in urls:

                all_urls.append({

                    "url": url,

                    "archive_date":
                    dt.strftime("%Y-%m-%d")

                })


        print("\nurls found\n")

        print(len(all_urls))


        # deduplicate

        seen = set()

        unique_urls = []


        for item in all_urls:

            if item["url"] in seen:
                continue

            seen.add(item["url"])

            unique_urls.append(item)


        print("\nunique urls\n")

        print(len(unique_urls))


        # step 2
        # metadata extraction

        print("\nextracting metadata\n")


        tasks = []


        for item in unique_urls:

            task = asyncio.create_task(

                process_article(

                    session,
                    semaphore,
                    item["url"],
                    item["archive_date"],
                    pbar

                )

            )

            tasks.append(task)


        await asyncio.gather(*tasks)


    pbar.close()

    csv_file.close()


    print("\ncorpus saved")

    print(output_csv)

    print("\nfinished")


# jupyter

await main()

In [ ]:
import pandas as pd
import re


# load corpus

df = pd.read_csv(
    "elmundo_corpus.csv"
)


print("\ninput corpus\n")

print(len(df))


# clean text fields

df["title"] = (
    df["title"]
    .fillna("")
    .astype(str)
)

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
)


# combined text

df["combined"] = (

    df["title"]
    + " "
    + df["text"]

).str.lower()


# high recall ukraine-russia lexicon

patterns = [

    # countries and leaders

    r"\bucrani\w*",
    r"\brusi\w*",
    r"\bmoscu\w*",
    r"\bmosk\w*",
    r"\bkremlin\w*",
    r"\bputin\w*",
    r"\bzelens\w*",
    r"\bzelensk\w*",
    r"\bzelenski\w*",
    r"\bzelensky\w*",
    r"\bmedvedev\w*",
    r"\blavrov\w*",
    r"\bshoigu\w*",
    r"\bprigozh\w*",
    r"\bwagner\w*",
    r"\blukashenko\w*",
    r"\bbielorrus\w*",


    # territories and cities

    r"\bdonbas\w*",
    r"\bdonetsk\w*",
    r"\blugansk\w*",
    r"\bcrime\w*",
    r"\bsebastopol\w*",
    r"\bmariupol\w*",
    r"\bodesa\w*",
    r"\bkherson\w*",
    r"\bkhark\w*",
    r"\bkiev\w*",
    r"\bkyiv\w*",
    r"\bzapori\w*",
    r"\bchernobil\w*",
    r"\bchernihiv\w*",
    r"\bbakhmut\w*",


    # war terminology

    r"\binvasion\w*",
    r"\bofensiv\w*",
    r"\bcontraofensiv\w*",
    r"\bbombarde\w*",
    r"\bmisil\w*",
    r"\bdron\w*",
    r"\bataqu\w*",
    r"\bejercit\w*",
    r"\btrop\w*",
    r"\bmercenario\w*",
    r"\bmovilizacion\w*",
    r"\banexion\w*",
    r"\bfrent\w*",
    r"\btrincher\w*",
    r"\bocupacion\w*",
    r"\bdespliegue militar\w*",
    r"\barmament\w*",
    r"\bdefensa aerea\w*",


    # geopolitical

    r"\botan\w*",
    r"\bnato\w*",
    r"\bsancion\w*",
    r"\bunion europea\w*",
    r"\bue\w*",
    r"\barmas occidentales\w*",
    r"\bayuda militar\w*",
    r"\bpaquete militar\w*",
    r"\bcorredor humanitario\w*",
    r"\bcrisis energetica\w*",
    r"\bgas ruso\w*",
    r"\bpetroleo ruso\w*",
    r"\benergia rusa\w*",


    # nuclear

    r"\bnuclear\w*",
    r"\bamenaza nuclear\w*",
    r"\bcentral nuclear\w*",
    r"\bzaporizhzhia\w*",


    # refugees and civilians

    r"\brefugiad\w*",
    r"\bdesplazad\w*",
    r"\bevacuacion\w*",
    r"\bcivil\w*",


    # propaganda and cyber

    r"\bdesinformacion\w*",
    r"\bpropaganda rusa\w*",
    r"\bfake news\w*",
    r"\bciberguerra\w*",
    r"\bataque cibernetico\w*",

]


combined_pattern = "|".join(patterns)


# filter corpus

mask = df["combined"].str.contains(

    combined_pattern,
    regex=True,
    na=False

)

filtered = df[mask].copy()


print("\nfiltered corpus\n")

print(len(filtered))


# deduplicate

if "url" in filtered.columns:

    filtered = filtered.drop_duplicates(
        subset=["url"]
    )

elif "article_url" in filtered.columns:

    filtered = filtered.drop_duplicates(
        subset=["article_url"]
    )


print("\nafter dedup\n")

print(len(filtered))


# section distribution

print("\nsection distribution\n")


if "section" in filtered.columns:

    print(

        filtered["section"]
        .value_counts()

    )

elif "section_article" in filtered.columns:

    print(

        filtered["section_article"]
        .value_counts()

    )


# year distribution

print("\nyear distribution\n")


years = pd.to_datetime(

    filtered["published_time"],

    errors="coerce",

    utc=True

)


print(

    years.dt.year
    .value_counts()
    .sort_index()

)


# image statistics

if "image_count" in filtered.columns:

    print("\nimage count distribution\n")

    print(

        filtered["image_count"]
        .describe()

    )


# articles with images

if "image_count" in filtered.columns:

    filtered["has_images"] = (
        filtered["image_count"] > 0
    )

    print("\narticles with images\n")

    print(

        filtered["has_images"]
        .value_counts()

    )


# live pages

url_column = None

if "url" in filtered.columns:
    url_column = "url"

elif "article_url" in filtered.columns:
    url_column = "article_url"


if url_column is not None:

    filtered["is_live"] = (

        filtered[url_column]
        .str.contains(
            "directo",
            case=False,
            na=False
        )

    )

    print("\nlive pages\n")

    print(

        filtered["is_live"]
        .value_counts()

    )


# save corpus

filtered.to_csv(

    "elmundo_ukraine_russia_high_recall.csv",

    index=False,

    encoding="utf-8-sig"

)


print("\nfiltered corpus saved\n")

print("elmundo_ukraine_russia_high_recall.csv")


print("\nfinished")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter


# settings

DATASET = "elmundo_ukraine_russia_high_recall.csv"


# load

df = pd.read_csv(DATASET)


print("\ninput rows\n")

print(len(df))


# clean

df["title"] = (
    df["title"]
    .fillna("")
    .astype(str)
)

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
)


# datetime

df["published_time"] = pd.to_datetime(

    df["published_time"],

    errors="coerce",

    utc=True

)


df["year"] = (
    df["published_time"]
    .dt.year
)

df["month"] = (
    df["published_time"]
    .dt.to_period("M")
    .astype(str)
)


# article length

df["text_length"] = (
    df["text"]
    .str.len()
)

df["word_count"] = (
    df["text"]
    .str.split()
    .str.len()
)


# live pages

url_column = None

if "url" in df.columns:
    url_column = "url"

elif "article_url" in df.columns:
    url_column = "article_url"


df["is_live"] = (

    df[url_column]
    .str.contains(
        "directo",
        case=False,
        na=False
    )

)


# section column

section_column = None

if "section" in df.columns:
    section_column = "section"

elif "section_article" in df.columns:
    section_column = "section_article"


# diagnostics

print("\nsection distribution\n")

print(

    df[section_column]
    .value_counts()
    .head(20)

)


print("\nyear distribution\n")

print(

    df["year"]
    .value_counts()
    .sort_index()

)


print("\nlive pages\n")

print(

    df["is_live"]
    .value_counts()

)


print("\narticle length stats\n")

print(

    df["word_count"]
    .describe()

)


print("\nimage count stats\n")

print(

    df["image_count"]
    .describe()

)


# monthly publication peaks

monthly_counts = (

    df.groupby("month")
    .size()
    .sort_index()

)


print("\nmonthly peaks\n")

print(

    monthly_counts
    .sort_values(ascending=False)
    .head(20)

)


# top title words

stopwords = {

    "de",
    "la",
    "el",
    "los",
    "las",
    "y",
    "en",
    "del",
    "que",
    "un",
    "una",
    "con",
    "por",
    "para",
    "a",
    "al",
    "se",
    "su",
    "sus",
    "como",
    "tras",
    "sobre",
    "desde",
    "ante",
    "entre",
    "más",
    "menos"

}


all_words = []


for title in df["title"]:

    words = re.findall(

        r"\b\w+\b",

        title.lower()

    )

    words = [

        w for w in words

        if len(w) > 3
        and w not in stopwords

    ]

    all_words.extend(words)


word_counts = Counter(all_words)


print("\ntop title words\n")

for word, count in word_counts.most_common(50):

    print(word, count)


# image domains

if "image_urls" in df.columns:

    image_domains = []


    for row in df["image_urls"].dropna():

        try:

            urls = eval(row)

            for u in urls:

                m = re.search(

                    r"https?://([^/]+)",

                    str(u)

                )

                if m:
                    image_domains.append(
                        m.group(1)
                    )

        except:
            pass


    domain_counts = Counter(image_domains)


    print("\nimage domains\n")

    for domain, count in domain_counts.most_common(20):

        print(domain, count)


# plots

plt.figure(figsize=(14, 6))

monthly_counts.plot()

plt.title("monthly publication volume")

plt.xticks(rotation=90)

plt.tight_layout()

plt.show()


plt.figure(figsize=(10, 6))

df[section_column].value_counts().head(15).plot(
    kind="bar"
)

plt.title("top sections")

plt.tight_layout()

plt.show()


plt.figure(figsize=(10, 6))

df["year"].value_counts().sort_index().plot(
    kind="bar"
)

plt.title("year distribution")

plt.tight_layout()

plt.show()


# save diagnostics

monthly_counts.to_csv(
    "elmundo_monthly_counts.csv"
)


print("\ndiagnostics saved\n")

print("elmundo_monthly_counts.csv")


print("\nfinished")

In [ ]:
import pandas as pd
import requests
import os
import hashlib
import time
import random
import ast

from tqdm import tqdm
from urllib.parse import urlparse


# settings

INPUT_DATASET = "elmundo_ukraine_russia_high_recall.csv"

OUTPUT_DIR = "elmundo_images"

MAX_IMAGES_PER_ARTICLE = None

ONLY_FIRST_IMAGE = False

START_INDEX = 0

SLEEP_RANGE = (0.1, 0.4)

TIMEOUT = 30


# create folders

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# load dataset

df = pd.read_csv(
    INPUT_DATASET
)


print("\ninput rows\n")

print(len(df))


# parse image urls

import ast
import pandas as pd


def parse_images(x):

    # NaN
    if pd.isna(x):
        return []

    # уже список
    if isinstance(x, list):
        return x

    x = str(x).strip()

    if not x:
        return []

    # формат:
    # ['url1', 'url2']
    if x.startswith("["):

        try:
            result = ast.literal_eval(x)

            if isinstance(result, list):
                return result

        except Exception:
            pass

    # формат:
    # url1 | url2 | url3
    if "|" in x:

        return [
            url.strip()
            for url in x.split("|")
            if url.strip()
        ]

    # одна ссылка
    return [x]


df["image_urls"] = (
    df["image_urls"]
    .apply(parse_images)
)

print("\ncheck\n")

print("rows:", len(df))

print(
    "rows with images:",
    (df["image_urls"].str.len() > 0).sum()
)

print(
    "first row images:",
    len(df["image_urls"].iloc[0])
)

print(
    df["image_urls"].iloc[0][:5]
)


# records

downloaded = []

failed = []


# headers

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


# download loop

for idx, row in tqdm(

    df.iterrows(),

    total=len(df)

):

    if idx < START_INDEX:
        continue


    article_url = row.get(
        "url",
        row.get("article_url", "")
    )

    title = row.get("title", "")

    section = row.get(
        "section",
        row.get("section_article", "")
    )

    published_time = row.get(
        "published_time",
        ""
    )

    image_urls = row["image_urls"]


    if ONLY_FIRST_IMAGE:
        image_urls = image_urls[:1]


    if MAX_IMAGES_PER_ARTICLE is not None:

        image_urls = image_urls[
            :MAX_IMAGES_PER_ARTICLE
        ]


    for image_idx, image_url in enumerate(image_urls):

        try:

            parsed = urlparse(image_url)

            ext = os.path.splitext(
                parsed.path
            )[1].lower()


            if ext not in [

                ".jpg",
                ".jpeg",
                ".png",
                ".webp",
                ".gif"

            ]:

                ext = ".jpg"


            image_hash = hashlib.md5(

                image_url.encode()

            ).hexdigest()


            filename = (
                f"{image_hash}{ext}"
            )


            section_folder = os.path.join(
                OUTPUT_DIR,
                str(section)
            )


            os.makedirs(
                section_folder,
                exist_ok=True
            )


            filepath = os.path.join(
                section_folder,
                filename
            )


            if os.path.exists(filepath):

                downloaded.append({

                    "article_url": article_url,
                    "title": title,
                    "section": section,
                    "published_time": published_time,
                    "image_url": image_url,
                    "filepath": filepath,
                    "status": "already_exists"

                })

                continue


            response = requests.get(

                image_url,

                headers=HEADERS,

                timeout=TIMEOUT,

                stream=True

            )


            response.raise_for_status()


            with open(filepath, "wb") as f:

                for chunk in response.iter_content(
                    8192
                ):

                    f.write(chunk)


            downloaded.append({

                "article_url": article_url,
                "title": title,
                "section": section,
                "published_time": published_time,
                "image_url": image_url,
                "filepath": filepath,
                "status": "downloaded"

            })


            time.sleep(

                random.uniform(
                    *SLEEP_RANGE
                )

            )


        except Exception as e:

            failed.append({

                "article_url": article_url,
                "image_url": image_url,
                "error": str(e)

            })

            print("\ndownload error\n")

            print(image_url)

            print(e)


# save logs

downloaded_df = pd.DataFrame(
    downloaded
)

failed_df = pd.DataFrame(
    failed
)


downloaded_df.to_csv(

    "elmundo_downloaded_images.csv",

    index=False,

    encoding="utf-8-sig"

)


failed_df.to_csv(

    "elmundo_failed_images.csv",

    index=False,

    encoding="utf-8-sig"

)


# final stats

print("\ndownload finished\n")

print(

    "images downloaded:",

    len(downloaded_df)

)

print(

    "failed images:",

    len(failed_df)

)


print("\nfiles saved\n")

print("elmundo_downloaded_images.csv")

print("elmundo_failed_images.csv")


print("\nfinished")

In [ ]:
import pandas as pd
import ast
from collections import Counter

INPUT_DATASET = "elmundo_ukraine_russia_high_recall.csv"


def parse_images(x):

    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    x = str(x).strip()

    if not x:
        return []

    # старый формат:
    # ['url1', 'url2']
    if x.startswith("["):

        try:
            result = ast.literal_eval(x)

            if isinstance(result, list):
                return result

        except Exception:
            pass

    # новый формат:
    # url1 | url2 | url3
    if "|" in x:

        return [
            url.strip()
            for url in x.split("|")
            if url.strip()
        ]

    return [x]


print("\nloading dataset...\n")

df = pd.read_csv(INPUT_DATASET)

print("rows:", len(df))

df["image_urls"] = df["image_urls"].apply(parse_images)

print("\ncollecting urls...\n")

all_urls = []

for urls in df["image_urls"]:
    all_urls.extend(urls)

print("all image urls:", len(all_urls))
print("unique image urls:", len(set(all_urls)))

# ---------------------------------------
# remove trackers
# ---------------------------------------

BAD_PATTERNS = [
    "smetrics",
    "tradedoubler",
    "/b/ss/"
]

real_urls = []

for url in all_urls:

    if any(
        bad in url.lower()
        for bad in BAD_PATTERNS
    ):
        continue

    real_urls.append(url)

print("\nafter removing trackers\n")

print("real image urls:", len(real_urls))
print("unique real image urls:", len(set(real_urls)))

# ---------------------------------------
# count frequencies
# ---------------------------------------

url_counts = Counter(real_urls)

print("\nTOP 50 MOST COMMON IMAGES\n")

for url, count in url_counts.most_common(50):

    print(f"{count:>6}  {url}")

# ---------------------------------------
# duplicates > 5
# ---------------------------------------

duplicates_df = pd.DataFrame(
    [
        (url, count)
        for url, count in url_counts.items()
        if count > 5
    ],
    columns=[
        "image_url",
        "count"
    ]
)

duplicates_df = duplicates_df.sort_values(
    "count",
    ascending=False
)

print("\nimages appearing more than 5 times:\n")

print(len(duplicates_df))

print("\nTOP 100 DUPLICATES\n")

print(
    duplicates_df.head(100)
)

duplicates_df.to_csv(
    "elmundo_images_more_than_5_times.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nfile saved:\n")

print("elmundo_images_more_than_5_times.csv")

print("\nfinished")

In [ ]:
import pandas as pd

freq = pd.read_csv(
    "elmundo_images_more_than_5_times.csv"
)

more_than_5 = freq[
    freq["count"] > 5
]

print()
print("images appearing > 5 times")
print(len(more_than_5))

print()
print("top 5")
print(
    more_than_5
)

In [ ]:
import pandas as pd
from urllib.parse import urlparse

# -------------------------
# load
# -------------------------

df = pd.read_csv(
    "elmundo_ukraine_russia_high_recall.csv"
)

duplicates = pd.read_csv(
    "elmundo_images_more_than_5_times.csv"
)

bad_urls = set(
    duplicates["image_url"]
)

# -------------------------
# domains to remove
# -------------------------

REMOVE_DOMAINS = {

    "smetrics.el-mundo.net",

    "impes.tradedoubler.com",

    "s1.dmcdn.net",
    "s2.dmcdn.net",

    "i.ytimg.com",
    "i9.ytimg.com",

    "datawrapper.dwcdn.net",

    "public.flourish.studio"

}

# -------------------------
# collect urls
# -------------------------

all_urls = []

for x in df["image_urls"].dropna():

    urls = [

        u.strip()

        for u in str(x).split("|")

        if u.strip()

    ]

    all_urls.extend(urls)

print()
print("all urls")
print(len(all_urls))

# -------------------------
# unique
# -------------------------

unique_urls = set(all_urls)

print()
print("unique urls")
print(len(unique_urls))

# -------------------------
# filtering
# -------------------------

clean_urls = []

removed_duplicates = 0
removed_domains = 0

for url in unique_urls:

    if url in bad_urls:

        removed_duplicates += 1
        continue

    try:

        domain = urlparse(url).netloc

    except:

        continue

    if domain in REMOVE_DOMAINS:

        removed_domains += 1
        continue

    clean_urls.append(url)

# -------------------------
# stats
# -------------------------

print()
print("removed >5 times")
print(removed_duplicates)

print()
print("removed bad domains")
print(removed_domains)

print()
print("final urls to download")
print(len(clean_urls))

# -------------------------
# save
# -------------------------

pd.DataFrame({

    "image_url": clean_urls

}).to_csv(

    "elmundo_unique_images_to_download.csv",

    index=False,

    encoding="utf-8-sig"

)

print()
print("saved")

print(
    "elmundo_unique_images_to_download.csv"
)

In [ ]:
import pandas as pd
import requests
import hashlib
import os
import time
import random

# =====================================
# SETTINGS
# =====================================

INPUT_FILE = (
    "elmundo_unique_images_to_download.csv"
)

OUTPUT_DIR = (
    "elmundo_unique_images"
)

FAILED_FILE = (
    "elmundo_failed_images.csv"
)

TIMEOUT = (10, 60)

SLEEP_RANGE = (0.3, 0.8)

# =====================================
# LOAD URLS
# =====================================

df = pd.read_csv(INPUT_FILE)

urls = (
    df["image_url"]
    .dropna()
    .astype(str)
    .unique()
)

print()
print("urls to download")
print(len(urls))

# =====================================
# FOLDER
# =====================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# =====================================
# SESSION
# =====================================

session = requests.Session()

headers = {

    "User-Agent":
    (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}

# =====================================
# STATS
# =====================================

downloaded = 0
skipped = 0

failed = []

# =====================================
# DOWNLOAD LOOP
# =====================================

for i, url in enumerate(urls):

    if i % 100 == 0:

        print(

            f"{i}/{len(urls)} "

            f"downloaded={downloaded} "

            f"skipped={skipped} "

            f"failed={len(failed)}"

        )

    try:

        ext = os.path.splitext(
            url.split("?")[0]
        )[1].lower()

        if ext not in [

            ".jpg",
            ".jpeg",
            ".png",
            ".webp",
            ".gif"

        ]:

            ext = ".jpg"

        filename = (

            hashlib.md5(
                url.encode("utf-8")
            ).hexdigest()

            + ext

        )

        filepath = os.path.join(
            OUTPUT_DIR,
            filename
        )

        # already exists

        if os.path.exists(filepath):

            skipped += 1
            continue

        response = session.get(

            url,

            headers=headers,

            timeout=TIMEOUT,

            stream=True

        )

        if response.status_code != 200:

            failed.append({

                "image_url": url,

                "status_code":
                response.status_code

            })

            continue

        with open(
            filepath,
            "wb"
        ) as f:

            for chunk in response.iter_content(
                chunk_size=8192
            ):

                if chunk:

                    f.write(chunk)

        downloaded += 1

        time.sleep(

            random.uniform(
                *SLEEP_RANGE
            )

        )

    except Exception as e:

        failed.append({

            "image_url": url,

            "error": str(e)

        })

        time.sleep(2)

# =====================================
# SAVE FAILS
# =====================================

failed_df = pd.DataFrame(
    failed
)

failed_df.to_csv(

    FAILED_FILE,

    index=False,

    encoding="utf-8-sig"

)

# =====================================
# FINAL
# =====================================

print()
print("downloaded")
print(downloaded)

print()
print("skipped")
print(skipped)

print()
print("failed")
print(len(failed))

print()
print("failed file")
print(FAILED_FILE)

print()
print("finished")

In [ ]:
import os

IMAGE_DIR = "elmundo_unique_images"

files = []

for f in os.listdir(IMAGE_DIR):
    if os.path.isfile(os.path.join(IMAGE_DIR, f)):
        files.append(f)

print("files on disk")
print(len(files))

In [ ]:
import os
import pandas as pd

IMAGE_DIR = "elmundo_unique_images"

rows = []

for fname in os.listdir(IMAGE_DIR):

    path = os.path.join(IMAGE_DIR, fname)

    if not os.path.isfile(path):
        continue

    size = os.path.getsize(path)

    rows.append({
        "file": fname,
        "size_bytes": size
    })

df = pd.DataFrame(rows)

print(df["size_bytes"].describe())

print()
print("smaller than 1 KB")
print((df["size_bytes"] < 1024).sum())

print()
print("smaller than 5 KB")
print((df["size_bytes"] < 5000).sum())

In [ ]:
from PIL import Image
import os
import pandas as pd

IMAGE_DIR = "elmundo_unique_images"

bad = []

files = os.listdir(IMAGE_DIR)

for i, fname in enumerate(files):

    if i % 5000 == 0:
        print(i)

    path = os.path.join(
        IMAGE_DIR,
        fname
    )

    try:

        img = Image.open(path)
        img.verify()

    except Exception as e:

        bad.append({
            "file": fname,
            "error": str(e)
        })

bad_df = pd.DataFrame(bad)

print()
print("corrupted")
print(len(bad_df))

bad_df.to_csv(
    "elmundo_corrupted_images.csv",
    index=False
)

In [ ]:
import pandas as pd

failed = pd.read_csv(
    "elmundo_failed_images.csv"
)

print("failed")
print(len(failed))

print()

print(failed.head())

print()

for col in failed.columns:

    print(col)

In [ ]:
import pandas as pd
import os
import hashlib

failed = pd.read_csv(
    "elmundo_failed_images.csv"
)

IMAGE_DIR = (
    "elmundo_unique_images"
)

exists = 0
missing = 0

for url in failed["image_url"]:

    ext = os.path.splitext(
        str(url).split("?")[0]
    )[1].lower()

    if ext not in [
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".gif"
    ]:
        ext = ".jpg"

    filename = (
        hashlib.md5(
            str(url).encode("utf-8")
        ).hexdigest()
        + ext
    )

    path = os.path.join(
        IMAGE_DIR,
        filename
    )

    if os.path.exists(path):
        exists += 1
    else:
        missing += 1

print()
print("failed urls")
print(len(failed))

print()
print("already on disk")
print(exists)

print()
print("actually missing")
print(missing)

In [ ]:
import pandas as pd

failed = pd.read_csv(
    "elmundo_failed_images.csv"
)

print(
    failed["status_code"]
    .value_counts(dropna=False)
)

In [ ]:
from urllib.parse import urlparse
import pandas as pd

failed["domain"] = failed[
    "image_url"
].apply(
    lambda x: urlparse(str(x)).netloc
)

print(
    failed["domain"]
    .value_counts()
)

In [ ]:
import pandas as pd
import requests
import os
import hashlib
import time
import random

from urllib.parse import urlparse
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


failed = pd.read_csv(
    "elmundo_failed_images.csv"
)

urls = (
    failed["image_url"]
    .dropna()
    .astype(str)
    .unique()
)

print()
print("urls to retry")
print(len(urls))


session = requests.Session()

retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2
)

adapter = HTTPAdapter(
    max_retries=retry
)

session.mount(
    "http://",
    adapter
)

session.mount(
    "https://",
    adapter
)


headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


os.makedirs(
    "elmundo_unique_images",
    exist_ok=True
)


downloaded = 0

skipped = 0

failed_again = []


for i, url in enumerate(urls):

    if i % 100 == 0:

        print(
            f"{i}/{len(urls)} "
            f"downloaded={downloaded} "
            f"skipped={skipped} "
            f"failed={len(failed_again)}"
        )

    try:

        ext = os.path.splitext(
            urlparse(url).path
        )[1].lower()

        if ext not in [
            ".jpg",
            ".jpeg",
            ".png",
            ".webp",
            ".gif"
        ]:

            ext = ".jpg"

        fname = (
            hashlib.md5(
                url.encode()
            ).hexdigest()
            + ext
        )

        path = os.path.join(
            "elmundo_unique_images",
            fname
        )

        if os.path.exists(path):

            skipped += 1
            continue

        r = session.get(

            url,

            headers=headers,

            timeout=120,

            stream=True

        )

        if r.status_code != 200:

            failed_again.append({

                "image_url": url,

                "status_code": r.status_code

            })

            continue

        with open(path, "wb") as f:

            for chunk in r.iter_content(
                65536
            ):

                if chunk:
                    f.write(chunk)

        downloaded += 1

        time.sleep(
            random.uniform(
                0.3,
                0.8
            )
        )

    except Exception as e:

        failed_again.append({

            "image_url": url,

            "error": str(e)

        })


pd.DataFrame(
    failed_again
).to_csv(

    "elmundo_failed_images_retry2.csv",

    index=False,

    encoding="utf-8-sig"

)


print()

print("downloaded")
print(downloaded)

print()

print("skipped")
print(skipped)

print()

print("failed")
print(len(failed_again))

print()

print("saved")
print("elmundo_failed_images_retry2.csv")

# El Diario

In [ ]:
import requests
import re

from collections import Counter
from bs4 import BeautifulSoup


# settings

target_years = [
    2022,
    2023,
    2024,
    2025,
    2026
]

sample_articles_per_year = 5


# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0 Safari/537.36"
    )
}


# session

session = requests.Session()

session.headers.update(
    headers
)


# sitemap index

sitemap_index_url = (
    "https://www.eldiario.es/"
    "sitemap_index_25b87.xml"
)


print("\nchecking sitemap index\n")

print(sitemap_index_url)


r = session.get(
    sitemap_index_url,
    timeout=30
)


print("\nstatus\n")

print(r.status_code)


xml = r.text


# extract sitemap urls

sitemap_urls = re.findall(

    r"<loc>(.*?)</loc>",

    xml
)


print("\ntotal sitemap urls found\n")

print(len(sitemap_urls))


# keep content sitemaps only

content_sitemaps = []

for s in sitemap_urls:

    if "sitemap_contents_" in s:

        content_sitemaps.append(s)


print("\ncontent sitemaps\n")

print(len(content_sitemaps))


# diagnostics containers

all_categories = Counter()

year_article_counts = Counter()

sample_article_urls = []


# process sitemaps

for sitemap_url in content_sitemaps:

    print("\n======================")
    print(sitemap_url)

    # detect year

    year_match = re.search(

        r"sitemap_contents_(\d{4})",

        sitemap_url
    )

    if not year_match:
        continue

    year = int(
        year_match.group(1)
    )

    if year not in target_years:
        continue

    try:

        r = session.get(
            sitemap_url,
            timeout=30
        )

        if r.status_code != 200:

            print("failed")

            continue

        sitemap_xml = r.text

        article_urls = re.findall(

            r"<loc>(.*?)</loc>",

            sitemap_xml
        )

        print(
            f"urls found: "
            f"{len(article_urls)}"
        )

        year_article_counts[year] += (
            len(article_urls)
        )

        # category extraction

        for u in article_urls:

            parts = u.split("/")

            if len(parts) > 3:

                category = parts[3]

                all_categories[
                    category
                ] += 1

        # collect samples

        sample_article_urls.extend(

            article_urls[
                :sample_articles_per_year
            ]
        )

    except Exception as e:

        print(e)


# summary

print("\n======================")
print("year distribution")
print("======================\n")

for year in sorted(
    year_article_counts
):

    print(
        year,
        year_article_counts[year]
    )


# categories

print("\n======================")
print("top categories")
print("======================\n")

for cat, count in all_categories.most_common(40):

    print(
        f"{cat:<35} {count}"
    )


# article diagnostics

print("\n======================")
print("article extraction test")
print("======================")


for article_url in sample_article_urls[:20]:

    print("\n----------------------")
    print(article_url)

    try:

        r = session.get(
            article_url,
            timeout=30
        )

        print(
            f"\nstatus: "
            f"{r.status_code}"
        )

        html = r.text

        # anti-bot detection

        anti_bot = False

        if (
            "captcha" in html.lower()
            or
            "cloudflare" in html.lower()
            or
            "access denied" in html.lower()
        ):

            anti_bot = True

        print("\nanti bot detected\n")

        print(anti_bot)


        soup = BeautifulSoup(

            html,

            "lxml"
        )

        # title

        title = None

        meta_title = soup.find(

            "meta",

            property="og:title"
        )

        if meta_title:

            title = meta_title.get(
                "content"
            )

        print("\ntitle\n")

        print(title)


        # publication date

        published = None

        meta_date = soup.find(

            "meta",

            property=(
                "article:published_time"
            )
        )

        if meta_date:

            published = meta_date.get(
                "content"
            )

        print("\npublished\n")

        print(published)


        # author

        author = None

        meta_author = soup.find(

            "meta",

            attrs={
                "name": "author"
            }
        )

        if meta_author:

            author = meta_author.get(
                "content"
            )

        print("\nauthor\n")

        print(author)


        # images

        image_urls = []

        for img in soup.find_all("img"):

            src = img.get("src")

            if src:

                image_urls.append(src)

        image_urls = list(
            set(image_urls)
        )

        print("\nimages found\n")

        print(len(image_urls))


        if len(image_urls) > 0:

            print("\nfirst images\n")

            for img in image_urls[:5]:

                print(img)


        # text extraction

        paragraphs = []

        for p in soup.find_all("p"):

            text = p.get_text(
                strip=True
            )

            if len(text) > 40:

                paragraphs.append(
                    text
                )

        article_text = "\n".join(
            paragraphs
        )

        print("\ntext length\n")

        print(len(article_text))


        print("\ntext preview\n")

        print(article_text[:1000])


    except Exception as e:

        print("\nerror\n")

        print(e)

### Download

In [ ]:
import csv
import requests
import re
import time
import random

from bs4 import BeautifulSoup
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

from tqdm import tqdm


# settings

target_years = [
    2022,
    2023,
    2024,
    2025,
    2026
]

max_workers = 10

timeout = 30

sleep_range = (
    0.2,
    1.0
)


# output file

metadata_file = (
    "eldiario_metadata.csv"
)


# allowed categories

allowed_categories = [

    "internacional",
    "politica",
    "opinion",
    "economia",
    "sociedad",

    "igualdad",
    "derechos-sociales",
    "derechos-humanos",

    "desalambre",

    "energia",

    "trabajo",
    "empleo",

    "seguridad",

    "ultima-hora",

    "fact-check",
    "bulos",
    "datos",

    "elecciones",

    "union-europea",

    "estados-unidos",
    "rusia",
    "ucrania",
    "china",
    "oriente-medio",
    "africa",
    "america-latina",

    "justicia",

    "tribuna-abierta",
    "contracorriente",

    "feminismo",
    "lgtbi",

    "memoria-historica",

    "guerra-ucrania",

    "opinion-editorial",

    "ultimas-noticias",

    "hemeroteca",
]


# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0 Safari/537.36"
    )
}


# session

session = requests.Session()

session.headers.update(
    headers
)


# sitemap index

sitemap_index_url = (
    "https://www.eldiario.es/"
    "sitemap_index_25b87.xml"
)


print("\nloading sitemap index\n")


r = session.get(
    sitemap_index_url,
    timeout=timeout
)


xml = r.text


# extract sitemap urls

sitemap_urls = re.findall(

    r"<loc>(.*?)</loc>",

    xml
)


# keep content sitemaps

content_sitemaps = []

for s in sitemap_urls:

    if "sitemap_contents_" in s:

        content_sitemaps.append(s)


print("\ncontent sitemaps found\n")

print(len(content_sitemaps))


# collect article urls

all_urls = []


print("\ncollecting urls\n")


for sitemap_url in content_sitemaps:

    try:

        # extract year

        year_match = re.search(

            r"sitemap_contents_(\d{4})",

            sitemap_url
        )

        if not year_match:
            continue

        year = int(
            year_match.group(1)
        )

        if year not in target_years:
            continue

        print(f"\nyear {year}")

        print(sitemap_url)

        r = session.get(
            sitemap_url,
            timeout=timeout
        )

        if r.status_code != 200:
            continue

        sitemap_xml = r.text

        urls = re.findall(

            r"<loc>(.*?)</loc>",

            sitemap_xml
        )

        print(
            f"urls found: {len(urls)}"
        )

        for url in urls:

            parts = url.split("/")

            if len(parts) < 4:
                continue

            category = parts[3]

            if category not in allowed_categories:
                continue

            all_urls.append(url)

    except Exception as e:

        print(e)


# deduplicate

all_urls = sorted(
    set(all_urls)
)


print("\nfinal urls\n")

print(len(all_urls))


# metadata extraction

def extract_metadata(url):

    try:

        r = session.get(

            url,

            timeout=timeout
        )

        if r.status_code != 200:
            return None

        soup = BeautifulSoup(

            r.text,

            "lxml"
        )

        # title

        title = None

        meta_title = soup.find(

            "meta",

            property="og:title"
        )

        if meta_title:

            title = meta_title.get(
                "content"
            )

        # publication date

        published = None

        meta_date = soup.find(

            "meta",

            property=(
                "article:published_time"
            )
        )

        if meta_date:

            published = meta_date.get(
                "content"
            )

        # author

        author = None

        meta_author = soup.find(

            "meta",

            attrs={
                "name": "author"
            }
        )

        if meta_author:

            author = meta_author.get(
                "content"
            )

        # short text preview

        paragraphs = []

        article = soup.find("article")

        if article:

            for p in article.find_all("p"):

                text = p.get_text(
                    strip=True
                )

                if len(text) > 40:

                    paragraphs.append(
                        text
                    )

        text_preview = " ".join(
            paragraphs[:5]
        )

        # skip broken pages

        if (
            title is None
            or
            len(text_preview) < 200
        ):

            return None

        # delay

        time.sleep(

            random.uniform(
                *sleep_range
            )
        )

        return {

            "url":
            url,

            "category":
            url.split("/")[3],

            "title":
            title,

            "published":
            published,

            "author":
            author,

            "text_preview":
            text_preview,
        }

    except:

        return None


# create csv

with open(

    metadata_file,

    "w",

    newline="",

    encoding="utf-8-sig"

) as out_file:

    writer = csv.DictWriter(

        out_file,

        fieldnames=[

            "url",
            "category",
            "title",
            "published",
            "author",
            "text_preview",
        ]
    )

    writer.writeheader()

    print("\nextracting metadata\n")

    with ThreadPoolExecutor(

        max_workers=max_workers

    ) as executor:

        futures = {

            executor.submit(
                extract_metadata,
                url
            ): url

            for url
            in all_urls
        }

        pbar = tqdm(
            total=len(all_urls)
        )

        saved = 0

        for future in as_completed(
            futures
        ):

            result = future.result()

            if result is not None:

                writer.writerow(
                    result
                )

                out_file.flush()

                saved += 1

            pbar.update(1)

            pbar.set_description(
                f"saved {saved}"
            )

        pbar.close()


print("\nfinished\n")

print(metadata_file)

In [ ]:
# load corpus

df = pd.read_csv(
    "eldiario_metadata.csv"
)


print("\ncorpus size\n")

print(len(df))


# columns

print("\ncolumns\n")

print(df.columns.tolist())


# duplicates

duplicate_urls = (

    df["url"]
    .duplicated()
    .sum()

)

print("\nduplicate urls\n")

print(duplicate_urls)


# missing values

print("\nmissing values\n")

print(

    df.isna()
    .sum()

)


# category distribution

print("\ncategory distribution\n")

print(

    df["category"]
    .value_counts()
    .sort_values(
        ascending=False
    )

)


# year distribution

years = pd.to_datetime(

    df["published"],

    errors="coerce",
    utc=True

)

print("\nyear distribution\n")

print(

    years.dt.year
    .value_counts()
    .sort_index()

)


# title lengths

df["title_length"] = (

    df["title"]
    .fillna("")
    .astype(str)
    .str.len()

)

print("\ntitle length statistics\n")

print(

    df["title_length"]
    .describe()

)


# preview lengths

df["preview_length"] = (

    df["text_preview"]
    .fillna("")
    .astype(str)
    .str.len()

)

print("\npreview length statistics\n")

print(

    df["preview_length"]
    .describe()

)


# empty previews

empty_previews = (

    df["text_preview"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()

)

print("\nempty previews\n")

print(empty_previews)


# shortest previews

print("\nshortest previews\n")

print(

    df.sort_values(
        "preview_length"
    )[
        [
            "category",
            "title",
            "preview_length",
            "url"
        ]
    ].head(20)

)


# longest previews

print("\nlongest previews\n")

print(

    df.sort_values(
        "preview_length",
        ascending=False
    )[
        [
            "category",
            "title",
            "preview_length",
            "url"
        ]
    ].head(20)

)


# categories by year

print("\ncategories by year\n")

tmp = df.copy()

tmp["year"] = years.dt.year

print(

    pd.crosstab(

        tmp["year"],
        tmp["category"]

    )

)


print("\ndiagnostics finished")

In [ ]:
import pandas as pd
import requests
import re

from collections import Counter


# load metadata

df = pd.read_csv(
    "eldiario_metadata.csv"
)

print("\nmetadata corpus size\n")

print(len(df))


# categories used in collection

allowed_categories = [

    "internacional",
    "politica",
    "opinion",
    "economia",
    "sociedad",

    "igualdad",
    "derechos-sociales",
    "derechos-humanos",

    "desalambre",

    "energia",

    "trabajo",
    "empleo",

    "seguridad",

    "ultima-hora",

    "fact-check",
    "bulos",
    "datos",

    "elecciones",

    "union-europea",

    "estados-unidos",
    "rusia",
    "ucrania",
    "china",
    "oriente-medio",
    "africa",
    "america-latina",

    "justicia",

    "tribuna-abierta",
    "contracorriente",

    "feminismo",
    "lgtbi",

    "memoria-historica",

    "guerra-ucrania",

    "opinion-editorial",

    "ultimas-noticias",

    "hemeroteca",
]


# sitemap index

index_url = (
    "https://www.eldiario.es/"
    "sitemap_index_25b87.xml"
)

print("\nloading sitemap index\n")

xml = requests.get(
    index_url,
    timeout=30
).text


# sitemap urls

sitemap_urls = re.findall(

    r"<loc>(.*?)</loc>",

    xml
)

content_sitemaps = [

    x

    for x in sitemap_urls

    if "sitemap_contents_" in x
]


# count urls from sitemap

sitemap_urls_filtered = []

sitemap_category_counts = Counter()


for sitemap_url in content_sitemaps:

    try:

        xml = requests.get(

            sitemap_url,

            timeout=30

        ).text

        urls = re.findall(

            r"<loc>(.*?)</loc>",

            xml
        )

        for url in urls:

            parts = url.split("/")

            if len(parts) < 4:
                continue

            category = parts[3]

            if category not in allowed_categories:
                continue

            sitemap_urls_filtered.append(
                url
            )

            sitemap_category_counts[
                category
            ] += 1

    except:
        pass


# unique sitemap urls

sitemap_urls_filtered = set(
    sitemap_urls_filtered
)


print("\nurls in sitemap\n")

print(
    len(
        sitemap_urls_filtered
    )
)


# metadata urls

metadata_urls = set(

    df["url"]
    .astype(str)

)


print("\nurls in metadata csv\n")

print(
    len(
        metadata_urls
    )
)


# missing urls

missing_urls = (

    sitemap_urls_filtered
    -
    metadata_urls

)


print("\nmissing urls\n")

print(
    len(
        missing_urls
    )
)


print("\nsuccess rate\n")

print(

    round(

        100
        *
        len(metadata_urls)
        /
        len(sitemap_urls_filtered),

        2

    ),

    "%"

)


# category comparison

print("\n===================")
print("sitemap categories")
print("===================\n")

for k, v in sorted(

    sitemap_category_counts.items(),

    key=lambda x: x[1],

    reverse=True

):

    print(
        f"{k:<25} {v}"
    )


print("\n===================")
print("metadata categories")
print("===================\n")

print(

    df["category"]
    .value_counts()

)


print("\nfinished")

In [ ]:
import pandas as pd
import requests
import re
import random

from bs4 import BeautifulSoup


# load metadata urls

metadata = pd.read_csv(
    "eldiario_metadata.csv"
)

metadata_urls = set(
    metadata["url"]
)


# load sitemap urls

xml = requests.get(
    "https://www.eldiario.es/sitemap_index_25b87.xml"
).text

sitemaps = re.findall(
    r"<loc>(.*?)</loc>",
    xml
)

content_sitemaps = [

    s

    for s in sitemaps

    if "sitemap_contents_" in s
]


all_urls = []


for sitemap in content_sitemaps:

    xml = requests.get(
        sitemap
    ).text

    urls = re.findall(
        r"<loc>(.*?)</loc>",
        xml
    )

    all_urls.extend(urls)


missing = list(

    set(all_urls)
    -
    metadata_urls

)


print(
    "missing urls:",
    len(missing)
)


sample = random.sample(

    missing,

    50
)


for url in sample:

    try:

        r = requests.get(
            url,
            timeout=20
        )

        print(
            r.status_code,
            url
        )

    except Exception as e:

        print(e)

In [ ]:
import pandas as pd
import requests
import re
import csv
import time

from bs4 import BeautifulSoup
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)
from tqdm import tqdm


# settings

max_workers = 15

timeout = 30


# files

existing_metadata_file = (
    "eldiario_metadata.csv"
)

missing_articles_file = (
    "eldiario_missing_articles.csv"
)

merged_file = (
    "eldiario_full_corpus.csv"
)


# categories

allowed_categories = [

    "internacional",
    "politica",
    "opinion",
    "economia",
    "sociedad",

    "igualdad",
    "derechos-sociales",
    "derechos-humanos",

    "desalambre",

    "energia",

    "trabajo",
    "empleo",

    "seguridad",

    "ultima-hora",

    "fact-check",
    "bulos",
    "datos",

    "elecciones",

    "union-europea",

    "estados-unidos",
    "rusia",
    "ucrania",
    "china",
    "oriente-medio",
    "africa",
    "america-latina",

    "justicia",

    "tribuna-abierta",
    "contracorriente",

    "feminismo",
    "lgtbi",

    "memoria-historica",

    "guerra-ucrania",

    "opinion-editorial",

    "ultimas-noticias",

    "hemeroteca",
]


# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0 Safari/537.36"
    )
}


session = requests.Session()

session.headers.update(
    headers
)


# load existing metadata

existing = pd.read_csv(
    existing_metadata_file
)

existing_urls = set(
    existing["url"].astype(str)
)

print("\nalready collected\n")

print(len(existing_urls))


# collect all sitemap urls

print("\nloading sitemap urls\n")

index_xml = session.get(

    "https://www.eldiario.es/sitemap_index_25b87.xml",

    timeout=timeout

).text


sitemaps = re.findall(

    r"<loc>(.*?)</loc>",

    index_xml
)


content_sitemaps = [

    s

    for s in sitemaps

    if "sitemap_contents_" in s
]


all_urls = []


for sitemap_url in tqdm(

    content_sitemaps,

    desc="sitemaps"

):

    try:

        xml = session.get(

            sitemap_url,

            timeout=timeout

        ).text

        urls = re.findall(

            r"<loc>(.*?)</loc>",

            xml
        )

        for url in urls:

            parts = url.split("/")

            if len(parts) < 4:
                continue

            category = parts[3]

            if category not in allowed_categories:
                continue

            all_urls.append(url)

    except:
        pass


all_urls = set(all_urls)


print("\nurls in sitemap\n")

print(len(all_urls))


# find missing urls

missing_urls = list(

    all_urls
    -
    existing_urls

)


print("\nmissing urls\n")

print(len(missing_urls))


# save missing urls separately

pd.DataFrame({

    "url": missing_urls

}).to_csv(

    "eldiario_missing_urls.csv",

    index=False,

    encoding="utf-8-sig"

)


print(
    "\nsaved: eldiario_missing_urls.csv"
)


# article extraction

def scrape_article(url):

    try:

        r = session.get(

            url,

            timeout=timeout

        )

        if r.status_code != 200:

            return None

        soup = BeautifulSoup(

            r.text,

            "lxml"
        )

        # title

        title = ""

        tag = soup.find(

            "meta",

            property="og:title"
        )

        if tag:

            title = tag.get(
                "content",
                ""
            )

        # date

        published = ""

        tag = soup.find(

            "meta",

            property=(
                "article:published_time"
            )
        )

        if tag:

            published = tag.get(
                "content",
                ""
            )

        # author

        author = ""

        tag = soup.find(

            "meta",

            attrs={
                "name": "author"
            }
        )

        if tag:

            author = tag.get(
                "content",
                ""
            )

        # category

        category = ""

        parts = url.split("/")

        if len(parts) > 3:

            category = parts[3]

        # full text

        paragraphs = []

        for p in soup.find_all("p"):

            txt = p.get_text(

                " ",

                strip=True

            )

            if len(txt) > 40:

                paragraphs.append(
                    txt
                )

        full_text = "\n".join(
            paragraphs
        )

        return {

            "url":
            url,

            "category":
            category,

            "title":
            title,

            "published":
            published,

            "author":
            author,

            "text":
            full_text,
        }

    except:

        return None


# scrape missing articles

with open(

    missing_articles_file,

    "w",

    newline="",

    encoding="utf-8-sig"

) as f:

    writer = csv.DictWriter(

        f,

        fieldnames=[

            "url",
            "category",
            "title",
            "published",
            "author",
            "text",
        ]
    )

    writer.writeheader()

    saved = 0

    with ThreadPoolExecutor(

        max_workers=max_workers

    ) as executor:

        futures = {

            executor.submit(

                scrape_article,

                url

            ): url

            for url
            in missing_urls
        }

        pbar = tqdm(

            total=len(
                missing_urls
            ),

            desc="articles"
        )

        for future in as_completed(
            futures
        ):

            result = future.result()

            if result is not None:

                writer.writerow(
                    result
                )

                f.flush()

                saved += 1

            pbar.update(1)

            pbar.set_postfix(

                saved=saved

            )

    pbar.close()


print(
    "\nmissing articles saved"
)

print(
    missing_articles_file
)

In [ ]:
# merge

print(
    "\nmerging files"
)

missing_df = pd.read_csv(
    missing_articles_file
)

existing["text"] = ""

merged = pd.concat(

    [

        existing,

        missing_df

    ],

    ignore_index=True

)

merged = merged.drop_duplicates(

    subset=["url"]
)


merged.to_csv(

    merged_file,

    index=False,

    encoding="utf-8-sig"
)


print(
    "\nfinal corpus size"
)

print(
    len(merged)
)

print(
    "\nsaved:"
)

print(
    merged_file
)

In [ ]:
# load corpus

df = pd.read_csv(
    "eldiario_full_corpus.csv"
)


print("\ncorpus size\n")

print(len(df))


# columns

print("\ncolumns\n")

print(df.columns.tolist())


# duplicates

duplicate_urls = (

    df["url"]
    .duplicated()
    .sum()

)

print("\nduplicate urls\n")

print(duplicate_urls)


# missing values

print("\nmissing values\n")

print(

    df.isna()
    .sum()

)


# category distribution

print("\ncategory distribution\n")

print(

    df["category"]
    .value_counts()
    .sort_values(
        ascending=False
    )

)


# years

years = pd.to_datetime(

    df["published"],

    errors="coerce",
    utc=True

)

print("\nyear distribution\n")

print(

    years.dt.year
    .value_counts()
    .sort_index()

)


# title lengths

df["title_length"] = (

    df["title"]
    .fillna("")
    .astype(str)
    .str.len()

)

print("\ntitle length statistics\n")

print(

    df["title_length"]
    .describe()

)


# text lengths

df["text_length"] = (

    df["text"]
    .fillna("")
    .astype(str)
    .str.len()

)

print("\ntext length statistics\n")

print(

    df["text_length"]
    .describe()

)


# empty texts

empty_texts = (

    df["text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()

)

print("\nempty texts\n")

print(empty_texts)


# very short texts

short_texts = (

    df["text_length"] < 500
).sum()

print("\nvery short texts (<500 chars)\n")

print(short_texts)


# longest articles

print("\nlongest articles\n")

print(

    df.sort_values(

        "text_length",

        ascending=False

    )[
        [
            "category",
            "title",
            "text_length",
            "url"
        ]
    ].head(20)

)


# shortest articles

print("\nshortest articles\n")

print(

    df.sort_values(
        "text_length"
    )[
        [
            "category",
            "title",
            "text_length",
            "url"
        ]
    ].head(20)

)


# texts by category

print("\nmean text length by category\n")

print(

    df.groupby(
        "category"
    )["text_length"]
    .mean()
    .sort_values(
        ascending=False
    )

)


# missing publication dates

missing_dates = (

    df["published"]
    .isna()
    .sum()

)

print("\nmissing publication dates\n")

print(missing_dates)


# texts missing by category

print("\nempty texts by category\n")

tmp = df.copy()

tmp["is_empty"] = (

    tmp["text"]
    .fillna("")
    .str.strip()
    .eq("")
)

print(

    tmp.groupby(
        "category"
    )["is_empty"]
    .sum()
    .sort_values(
        ascending=False
    )

)


print("\ndiagnostics finished")

In [ ]:
# files

input_file = (
    "eldiario_full_corpus.csv"
)

output_file = (
    "eldiario_ukraine_russia_high_recall.csv"
)


# load corpus

df = pd.read_csv(
    input_file
)


print("\ninput corpus\n")

print(len(df))


# fill missing

for col in [

    "title",
    "text_preview",
    "text"

]:

    if col in df.columns:

        df[col] = (

            df[col]
            .fillna("")
            .astype(str)

        )


# combine text

df["combined_text"] = (

    df["title"]
    + " "
    + df["text_preview"]
    + " "
    + df["text"]

).str.lower()


# anchor patterns

anchor_patterns = [

    # ukraine

    r"\bucrani\w*",

    # russia

    r"\brusi\w*",
    r"\brus\w*",

    # putin

    r"\bputin\w*",

    # zelensky

    r"\bzelens\w*",
    r"\bzelensk\w*",
    r"\bzelenski\w*",
    r"\bzelensky\w*",

    # kremlin

    r"\bkremlin\w*",

    # moscow

    r"\bmoscu\w*",

]


# strong anchors

strong_anchor_patterns = [

    r"\bucrani\w*",
    r"\brusi\w*",
    r"\bputin\w*",
    r"\bzelens\w*",
    r"\bkremlin\w*",

]


# war / geopolitical context

context_patterns = [

    # war

    r"\bguerra\w*",
    r"\binvasion\w*",
    r"\bofensiv\w*",
    r"\bcontraofensiv\w*",

    # military

    r"\bmisil\w*",
    r"\bdron\w*",
    r"\bataqu\w*",
    r"\bejercit\w*",
    r"\btrop\w*",
    r"\bmercenario\w*",

    # territories

    r"\bdonbas\w*",
    r"\bdonetsk\w*",
    r"\blugansk\w*",
    r"\bcrime\w*",
    r"\bsebastopol\w*",
    r"\bmariupol\w*",
    r"\bodesa\w*",
    r"\bkherson\w*",
    r"\bkhark\w*",
    r"\bkiev\w*",
    r"\bkyiv\w*",
    r"\bbakhmut\w*",

    # geopolitics

    r"\botan\w*",
    r"\bnato\w*",
    r"\bsancion\w*",
    r"\bayuda militar\w*",
    r"\barmas occidentales\w*",

    # energy

    r"\bgas ruso\w*",
    r"\bpetroleo ruso\w*",
    r"\benergia rusa\w*",
    r"\bnord\s*stream\w*",

    # nuclear

    r"\bnuclear\w*",

]


# compile regex

anchor_regex = re.compile(

    "|".join(anchor_patterns),

    flags=re.IGNORECASE

)

strong_anchor_regex = re.compile(

    "|".join(
        strong_anchor_patterns
    ),

    flags=re.IGNORECASE

)

context_regex = re.compile(

    "|".join(context_patterns),

    flags=re.IGNORECASE

)


# filtering

anchor_match = (

    df["combined_text"]
    .str.contains(
        anchor_regex,
        na=False
    )

)

context_match = (

    df["combined_text"]
    .str.contains(
        context_regex,
        na=False
    )

)

strong_anchor_match = (

    df["combined_text"]
    .str.contains(
        strong_anchor_regex,
        na=False
    )

)


# final logic

mask = (

    (anchor_match & context_match)

    |

    strong_anchor_match

)


filtered = df[mask].copy()


# deduplicate

filtered = filtered.drop_duplicates(

    subset=["url"]

)


# diagnostics

print("\nfiltered corpus\n")

print(len(filtered))


print("\ncategory distribution\n")

print(

    filtered["category"]
    .value_counts()

)


# years

filtered["published"] = pd.to_datetime(

    filtered["published"],

    errors="coerce",

    utc=True

)


print("\nyear distribution\n")

print(

    filtered["published"]
    .dt.year
    .value_counts()
    .sort_index()

)


# text lengths

filtered["text_length"] = (

    filtered["text"]
    .fillna("")
    .astype(str)
    .str.len()

)


print("\ntext length statistics\n")

print(

    filtered["text_length"]
    .describe()

)


# articles with full text

filtered["has_full_text"] = (

    filtered["text_length"] > 500

)


print("\narticles with full text\n")

print(

    filtered["has_full_text"]
    .value_counts()

)


# sample titles

print("\nsample titles\n")

for title in filtered["title"].head(30):

    print(title)


# save

filtered.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nfiltered corpus saved\n")

print(output_file)


print("\nfinished")

### Text extraction

In [ ]:
import pandas as pd
import requests
import json
import time
import csv

from bs4 import BeautifulSoup
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)
from tqdm import tqdm


# =========================================
# SETTINGS
# =========================================

input_file = (
    "eldiario_full_corpus.csv"
)

output_file = (
    "eldiario_fulltext_recovered.csv"
)

failed_file = (
    "eldiario_fulltext_failed.csv"
)

max_workers = 20

timeout = 30

save_every = 100


# =========================================
# LOAD CORPUS
# =========================================

df = pd.read_csv(
    input_file
)


print("\ninput corpus\n")

print(len(df))


# make sure text column exists

if "text" not in df.columns:

    df["text"] = ""


df["text"] = (

    df["text"]
    .fillna("")
    .astype(str)

)


# only recover missing texts

missing_mask = (

    df["text"]
    .str.strip()
    .eq("")

)

missing_df = df[
    missing_mask
].copy()


print("\narticles missing text\n")

print(len(missing_df))


urls = missing_df["url"].tolist()


# =========================================
# REQUESTS SESSION
# =========================================

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0 Safari/537.36"
    )
}


session = requests.Session()

session.headers.update(
    headers
)


# =========================================
# FULL TEXT EXTRACTION
# =========================================

def extract_full_text(url):

    try:

        r = session.get(

            url,

            timeout=timeout

        )

        if r.status_code != 200:

            return {
                "url": url,
                "text": None,
                "status": r.status_code
            }

        soup = BeautifulSoup(

            r.text,

            "lxml"
        )

        # find all JSON-LD scripts

        scripts = soup.find_all(

            "script",

            type="application/ld+json"
        )

        for s in scripts:

            try:

                if not s.string:
                    continue

                data = json.loads(
                    s.string
                )

                # sometimes list

                if isinstance(data, list):

                    items = data

                else:

                    items = [data]

                for item in items:

                    if not isinstance(
                        item,
                        dict
                    ):
                        continue

                    article_type = item.get(
                        "@type",
                        ""
                    )

                    if isinstance(
                        article_type,
                        list
                    ):

                        article_type = " ".join(
                            article_type
                        )

                    # detect article

                    if any(

                        x in str(
                            article_type
                        )

                        for x in [

                            "NewsArticle",
                            "Article",
                            "ReportageNewsArticle"

                        ]
                    ):

                        body = item.get(
                            "articleBody",
                            ""
                        )

                        if (
                            body
                            and
                            len(body) > 200
                        ):

                            return {

                                "url": url,

                                "text": body,

                                "status": 200

                            }

            except:
                continue

        return {

            "url": url,

            "text": None,

            "status": "no_articleBody"

        }

    except Exception as e:

        return {

            "url": url,

            "text": None,

            "status": str(e)

        }


# =========================================
# SCRAPE TEXTS
# =========================================

results = []

failed = []


with ThreadPoolExecutor(

    max_workers=max_workers

) as executor:

    futures = {

        executor.submit(

            extract_full_text,

            url

        ): url

        for url in urls
    }

    pbar = tqdm(

        total=len(urls),

        desc="recovering texts"

    )

    success_count = 0

    for future in as_completed(
        futures
    ):

        result = future.result()

        if result["text"] is not None:

            results.append(result)

            success_count += 1

        else:

            failed.append(result)

        pbar.update(1)

        pbar.set_postfix(

            recovered=success_count

        )

        # periodic save

        if len(results) % save_every == 0:

            tmp = pd.DataFrame(
                results
            )

            tmp.to_csv(

                output_file,

                index=False,

                encoding="utf-8-sig"

            )

    pbar.close()


# =========================================
# SAVE RECOVERED TEXTS
# =========================================

recovered_df = pd.DataFrame(
    results
)

recovered_df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nrecovered texts\n")

print(len(recovered_df))


# =========================================
# SAVE FAILED URLS
# =========================================

failed_df = pd.DataFrame(
    failed
)

failed_df.to_csv(

    failed_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nfailed urls\n")

print(len(failed_df))


# =========================================
# MERGE BACK INTO MAIN CORPUS
# =========================================

print("\nmerging recovered texts\n")


text_map = dict(

    zip(

        recovered_df["url"],

        recovered_df["text"]

    )

)


updated = 0


for idx in df.index:

    url = df.at[idx, "url"]

    current_text = str(
        df.at[idx, "text"]
    )

    if (

        current_text.strip() == ""

        and

        url in text_map

    ):

        df.at[idx, "text"] = (
            text_map[url]
        )

        updated += 1


print("\nupdated rows\n")

print(updated)


# =========================================
# SAVE FINAL CORPUS
# =========================================

final_output = (
    "eldiario_full_corpus_with_fulltext.csv"
)

df.to_csv(

    final_output,

    index=False,

    encoding="utf-8-sig"

)


print("\nfinal corpus saved\n")

print(final_output)


# =========================================
# FINAL DIAGNOSTICS
# =========================================

text_lengths = (

    df["text"]
    .fillna("")
    .astype(str)
    .str.len()

)

print("\ntexts >500 chars\n")

print(
    (text_lengths > 500).sum()
)

print("\nempty texts\n")

print(
    (text_lengths == 0).sum()
)

print("\nfinished")

In [ ]:
import pandas as pd


# load final corpus

df = pd.read_csv(
    "eldiario_full_corpus_with_fulltext.csv"
)


print("\ncorpus size\n")

print(len(df))


# columns

print("\ncolumns\n")

print(df.columns.tolist())


# duplicates

duplicate_urls = (

    df["url"]
    .duplicated()
    .sum()

)

print("\nduplicate urls\n")

print(duplicate_urls)


# missing values

print("\nmissing values\n")

print(

    df.isna()
    .sum()

)


# category distribution

print("\ncategory distribution\n")

print(

    df["category"]
    .value_counts()
    .sort_values(
        ascending=False
    )

)


# years

years = pd.to_datetime(

    df["published"],

    errors="coerce",
    utc=True

)

print("\nyear distribution\n")

print(

    years.dt.year
    .value_counts()
    .sort_index()

)


# text lengths

df["text_length"] = (

    df["text"]
    .fillna("")
    .astype(str)
    .str.len()

)

print("\ntext length statistics\n")

print(

    df["text_length"]
    .describe()

)


# empty texts

empty_texts = (

    df["text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()

)

print("\nempty texts\n")

print(empty_texts)


# texts >500 chars

long_texts = (

    df["text_length"] > 500
).sum()

print("\ntexts >500 chars\n")

print(long_texts)


# texts >2000 chars

very_long_texts = (

    df["text_length"] > 2000
).sum()

print("\ntexts >2000 chars\n")

print(very_long_texts)


# shortest texts

print("\nshortest texts\n")

print(

    df.sort_values(
        "text_length"
    )[
        [
            "category",
            "title",
            "text_length",
            "url"
        ]
    ].head(20)

)


# longest texts

print("\nlongest texts\n")

print(

    df.sort_values(

        "text_length",

        ascending=False

    )[
        [
            "category",
            "title",
            "text_length",
            "url"
        ]
    ].head(20)

)


# mean text length by category

print("\nmean text length by category\n")

print(

    df.groupby(
        "category"
    )["text_length"]
    .mean()
    .sort_values(
        ascending=False
    )

)


# missing publication dates

missing_dates = (

    df["published"]
    .isna()
    .sum()

)

print("\nmissing publication dates\n")

print(missing_dates)


# texts missing by category

print("\nempty texts by category\n")

tmp = df.copy()

tmp["is_empty"] = (

    tmp["text"]
    .fillna("")
    .str.strip()
    .eq("")
)

print(

    tmp.groupby(
        "category"
    )["is_empty"]
    .sum()
    .sort_values(
        ascending=False
    )

)


print("\ndiagnostics finished")

In [ ]:
import pandas as pd
import requests
import json

from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


# =====================================
# FILES
# =====================================

input_file = "eldiario_full_corpus_with_fulltext.csv"

output_file = "eldiario_full_corpus_with_fulltext_v2.csv"

failed_file = "eldiario_text_recovery_failed_v2.csv"


# =====================================
# LOAD
# =====================================

df = pd.read_csv(input_file)

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
)

missing_mask = (

    df["text"]
    .str.strip()
    .eq("")

)

missing_df = df[
    missing_mask
].copy()

print("\nmissing texts\n")
print(len(missing_df))


# =====================================
# REQUESTS
# =====================================

session = requests.Session()

session.headers.update({

    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"

})


# =====================================
# RECURSIVE SEARCH
# =====================================

def find_article_body(obj):

    if isinstance(obj, dict):

        if "articleBody" in obj:

            body = obj["articleBody"]

            if isinstance(body, str):

                body = body.strip()

                if body:

                    return body

        for value in obj.values():

            result = find_article_body(value)

            if result:

                return result

    elif isinstance(obj, list):

        for item in obj:

            result = find_article_body(item)

            if result:

                return result

    return None


# =====================================
# EXTRACT
# =====================================

def recover_text(url):

    try:

        r = session.get(
            url,
            timeout=30
        )

        if r.status_code != 200:

            return (
                url,
                None,
                f"status_{r.status_code}"
            )

        soup = BeautifulSoup(
            r.text,
            "lxml"
        )

        scripts = soup.find_all(
            "script",
            type="application/ld+json"
        )

        for script in scripts:

            try:

                if not script.string:
                    continue

                data = json.loads(
                    script.string
                )

                body = find_article_body(
                    data
                )

                if body:

                    return (
                        url,
                        body,
                        "success"
                    )

            except Exception:
                continue

        return (
            url,
            None,
            "no_articleBody"
        )

    except Exception as e:

        return (
            url,
            None,
            str(e)
        )


# =====================================
# RECOVERY
# =====================================

results = []
failed = []

urls = missing_df["url"].tolist()

with ThreadPoolExecutor(
    max_workers=10
) as executor:

    futures = {

        executor.submit(
            recover_text,
            url
        ): url

        for url in urls

    }

    for future in tqdm(

        as_completed(futures),

        total=len(futures),

        desc="recovering texts"

    ):

        url, text, status = (
            future.result()
        )

        if text:

            results.append(
                (url, text)
            )

        else:

            failed.append(
                (url, status)
            )


# =====================================
# SAVE RECOVERED
# =====================================

recovered_df = pd.DataFrame(

    results,

    columns=[
        "url",
        "recovered_text"
    ]

)

recovered_df.to_csv(

    "eldiario_recovered_texts_v2.csv",

    index=False,

    encoding="utf-8-sig"

)


# =====================================
# SAVE FAILED
# =====================================

failed_df = pd.DataFrame(

    failed,

    columns=[
        "url",
        "reason"
    ]

)

failed_df.to_csv(

    failed_file,

    index=False,

    encoding="utf-8-sig"

)


# =====================================
# MERGE
# =====================================

text_map = dict(

    zip(

        recovered_df["url"],

        recovered_df[
            "recovered_text"
        ]

    )

)

updated = 0

for idx in df.index:

    current_text = str(
        df.at[idx, "text"]
    ).strip()

    if not current_text:

        url = df.at[idx, "url"]

        if url in text_map:

            df.at[idx, "text"] = (
                text_map[url]
            )

            updated += 1


print("\nupdated rows\n")
print(updated)


# =====================================
# SAVE FINAL
# =====================================

df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)

print("\nfinal corpus saved\n")
print(output_file)

print("\nfailed urls\n")
print(len(failed_df))

print("\nfinished")

In [ ]:
import pandas as pd
import requests
import json

from bs4 import BeautifulSoup
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)
from tqdm import tqdm


# =====================================
# SETTINGS
# =====================================

input_file = (
    "eldiario_full_corpus_with_fulltext.csv"
)

failed_file = (
    "eldiario_text_recovery_failed.csv"
)

max_workers = 8

timeout = 35

save_every = 500


# =====================================
# LOAD CORPUS
# =====================================

df = pd.read_csv(
    input_file
)

print("\ncorpus size\n")
print(len(df))

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
)

missing_mask = (

    df["text"]
    .str.strip()
    .eq("")

)

missing_urls = (

    df.loc[
        missing_mask,
        "url"
    ]
    .dropna()
    .tolist()

)

print("\nmissing texts\n")
print(len(missing_urls))


# =====================================
# SESSION
# =====================================

session = requests.Session()

session.headers.update({

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122 Safari/537.36"
    )

})


# =====================================
# RECURSIVE SEARCH
# =====================================

def find_article_body(obj):

    if isinstance(obj, dict):

        if "articleBody" in obj:

            body = obj["articleBody"]

            if isinstance(
                body,
                str
            ):

                body = body.strip()

                if body:

                    return body

        for value in obj.values():

            result = find_article_body(
                value
            )

            if result:

                return result

    elif isinstance(obj, list):

        for item in obj:

            result = find_article_body(
                item
            )

            if result:

                return result

    return None


# =====================================
# RECOVER TEXT
# =====================================

def recover_text(url):

    try:

        r = session.get(

            url,

            timeout=timeout

        )

        if r.status_code != 200:

            return (
                url,
                None,
                f"status_{r.status_code}"
            )

        soup = BeautifulSoup(

            r.text,

            "lxml"
        )

        scripts = soup.find_all(

            "script",

            type="application/ld+json"

        )

        for script in scripts:

            try:

                if not script.string:
                    continue

                data = json.loads(
                    script.string
                )

                body = find_article_body(
                    data
                )

                if body:

                    return (
                        url,
                        body,
                        "success"
                    )

            except:
                continue

        return (

            url,

            None,

            "no_articleBody"

        )

    except Exception as e:

        return (

            url,

            None,

            str(e)

        )


# =====================================
# RECOVERY LOOP
# =====================================

failed = []

updated = 0


with ThreadPoolExecutor(

    max_workers=max_workers

) as executor:

    futures = {

        executor.submit(
            recover_text,
            url
        ): url

        for url in missing_urls

    }

    for future in tqdm(

        as_completed(futures),

        total=len(futures),

        desc="recovering texts"

    ):

        url, text, status = (

            future.result()

        )

        if text:

            mask = (

                df["url"]
                == url

            )

            df.loc[
                mask,
                "text"
            ] = text

            updated += 1

            if (

                updated > 0

                and

                updated % save_every == 0

            ):

                df.to_csv(

                    input_file,

                    index=False,

                    encoding="utf-8-sig"

                )

                print(

                    f"\nprogress saved: {updated}"

                )

        else:

            failed.append(

                {
                    "url": url,
                    "reason": status
                }

            )


# =====================================
# FINAL SAVE
# =====================================

print("\nfinal save\n")

df.to_csv(

    input_file,

    index=False,

    encoding="utf-8-sig"

)


# =====================================
# FAILED URLS
# =====================================

failed_df = pd.DataFrame(
    failed
)

failed_df.to_csv(

    failed_file,

    index=False,

    encoding="utf-8-sig"

)


# =====================================
# SUMMARY
# =====================================

print("\nupdated texts\n")
print(updated)

print("\nfailed urls\n")
print(len(failed_df))

print("\nmain corpus updated\n")
print(input_file)

print("\nfinished")

In [ ]:
import pandas as pd

df = pd.read_csv(
    "eldiario_text_recovery_failed.csv"
)

print("\nfailed urls\n")
print(len(df))

print("\nreason distribution\n")

print(
    df["reason"]
    .value_counts()
    .head(50)
)

In [ ]:
import pandas as pd
import re


# =====================================
# FILES
# =====================================

input_file = (
    "eldiario_full_corpus_with_fulltext.csv"
)

output_file = (
    "eldiario_ukraine_russia_full_filtered.csv"
)


# =====================================
# LOAD CORPUS
# =====================================

df = pd.read_csv(
    input_file
)


print("\ninput corpus\n")

print(len(df))


# =====================================
# CLEAN TEXT FIELDS
# =====================================

for col in [

    "title",
    "text_preview",
    "text"

]:

    if col in df.columns:

        df[col] = (

            df[col]
            .fillna("")
            .astype(str)

        )


# =====================================
# COMBINED TEXT
# =====================================

df["combined_text"] = (

    df["title"]
    + " "
    + df["text_preview"]
    + " "
    + df["text"]

).str.lower()


# =====================================
# ANCHOR KEYWORDS
# =====================================

anchor_patterns = [

    # ukraine

    r"\bucrani\w*",

    # russia

    r"\brusi\w*",
    r"\brus\w*",

    # putin

    r"\bputin\w*",

    # zelensky

    r"\bzelens\w*",
    r"\bzelensk\w*",
    r"\bzelenski\w*",
    r"\bzelensky\w*",

    # kremlin / moscow

    r"\bkremlin\w*",
    r"\bmoscu\w*",

]


# =====================================
# STRONG ANCHORS
# =====================================

strong_anchor_patterns = [

    r"\bucrani\w*",
    r"\brusi\w*",
    r"\bputin\w*",
    r"\bzelens\w*",
    r"\bkremlin\w*",

]


# =====================================
# CONTEXT KEYWORDS
# =====================================

context_patterns = [

    # war

    r"\bguerra\w*",
    r"\binvasion\w*",
    r"\bofensiv\w*",
    r"\bcontraofensiv\w*",

    # military

    r"\bmisil\w*",
    r"\bdron\w*",
    r"\bataqu\w*",
    r"\bejercit\w*",
    r"\btrop\w*",
    r"\bmercenario\w*",

    # territories

    r"\bdonbas\w*",
    r"\bdonetsk\w*",
    r"\blugansk\w*",
    r"\bcrime\w*",
    r"\bsebastopol\w*",
    r"\bmariupol\w*",
    r"\bodesa\w*",
    r"\bkherson\w*",
    r"\bkhark\w*",
    r"\bkiev\w*",
    r"\bkyiv\w*",
    r"\bbakhmut\w*",

    # geopolitics

    r"\botan\w*",
    r"\bnato\w*",
    r"\bsancion\w*",
    r"\bayuda militar\w*",
    r"\barmas occidentales\w*",

    # energy

    r"\bgas ruso\w*",
    r"\bpetroleo ruso\w*",
    r"\benergia rusa\w*",
    r"\bnord\s*stream\w*",

    # nuclear

    r"\bnuclear\w*",

    # refugees

    r"\brefugiad\w*",
    r"\bevacuacion\w*",

]


# =====================================
# COMPILE REGEX
# =====================================

anchor_regex = re.compile(

    "|".join(anchor_patterns),

    flags=re.IGNORECASE

)

strong_anchor_regex = re.compile(

    "|".join(
        strong_anchor_patterns
    ),

    flags=re.IGNORECASE

)

context_regex = re.compile(

    "|".join(context_patterns),

    flags=re.IGNORECASE

)


# =====================================
# MATCHES
# =====================================

anchor_match = (

    df["combined_text"]
    .str.contains(
        anchor_regex,
        na=False
    )

)

context_match = (

    df["combined_text"]
    .str.contains(
        context_regex,
        na=False
    )

)

strong_anchor_match = (

    df["combined_text"]
    .str.contains(
        strong_anchor_regex,
        na=False
    )

)


# =====================================
# FINAL FILTER
# =====================================

mask = (

    (anchor_match & context_match)

    |

    strong_anchor_match

)


filtered = df[
    mask
].copy()


# =====================================
# DEDUPLICATION
# =====================================

filtered = filtered.drop_duplicates(

    subset=["url"]

)


# =====================================
# DIAGNOSTICS
# =====================================

print("\nfiltered corpus\n")

print(len(filtered))


# category distribution

print("\ncategory distribution\n")

print(

    filtered["category"]
    .value_counts()

)


# year distribution

filtered["published"] = pd.to_datetime(

    filtered["published"],

    errors="coerce",

    utc=True

)

print("\nyear distribution\n")

print(

    filtered["published"]
    .dt.year
    .value_counts()
    .sort_index()

)


# text lengths

filtered["text_length"] = (

    filtered["text"]
    .fillna("")
    .astype(str)
    .str.len()

)

print("\ntext length statistics\n")

print(

    filtered["text_length"]
    .describe()

)


# articles with full text

filtered["has_full_text"] = (

    filtered["text_length"] > 500

)

print("\narticles with full text\n")

print(

    filtered["has_full_text"]
    .value_counts()

)


# longest texts

print("\nlongest texts\n")

print(

    filtered.sort_values(

        "text_length",

        ascending=False

    )[
        [
            "category",
            "title",
            "text_length"
        ]
    ].head(20)

)


# sample titles

print("\nsample titles\n")

for title in filtered["title"].head(50):

    print(title)


# =====================================
# SAVE
# =====================================

filtered.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nfiltered corpus saved\n")

print(output_file)


print("\nfinished")

In [ ]:
import pandas as pd
import requests
import re

from bs4 import BeautifulSoup
from tqdm import tqdm


# =====================================
# FILES
# =====================================

input_file = (
    "eldiario_ukraine_russia_full_filtered.csv"
)

output_file = (
    "eldiario_ukraine_russia_with_images.csv"
)


# =====================================
# LOAD FILTERED CORPUS
# =====================================

df = pd.read_csv(
    input_file
)


print("\nfiltered corpus\n")

print(len(df))


# =====================================
# URL SET
# =====================================

target_urls = set(

    df["url"]
    .dropna()
    .astype(str)

)


# =====================================
# REQUESTS
# =====================================

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0 Safari/537.36"
    )
}


session = requests.Session()

session.headers.update(
    headers
)


# =====================================
# LOAD SITEMAP INDEX
# =====================================

print("\nloading sitemap index\n")

index_xml = session.get(

    "https://www.eldiario.es/sitemap_index_25b87.xml",

    timeout=30

).text


sitemaps = re.findall(

    r"<loc>(.*?)</loc>",

    index_xml

)


content_sitemaps = [

    s

    for s in sitemaps

    if "sitemap_contents_" in s
]


print("\ncontent sitemaps\n")

print(len(content_sitemaps))


# =====================================
# IMAGE STORAGE
# =====================================

image_map = {}


# =====================================
# PARSE SITEMAPS
# =====================================

for sitemap_url in tqdm(

    content_sitemaps,

    desc="parsing sitemaps"

):

    try:

        xml = session.get(

            sitemap_url,

            timeout=30

        ).text

        soup = BeautifulSoup(

            xml,

            "xml"
        )

        urls = soup.find_all("url")

        for u in urls:

            loc = u.find("loc")

            if not loc:
                continue

            article_url = (
                loc.text.strip()
            )

            if article_url not in target_urls:
                continue

            images = u.find_all(
                "image:image"
            )

            image_urls = []

            image_captions = []

            for img in images:

                img_loc = img.find(
                    "image:loc"
                )

                if img_loc:

                    image_urls.append(
                        img_loc.text.strip()
                    )

                img_title = img.find(
                    "image:title"
                )

                if img_title:

                    image_captions.append(
                        img_title.text.strip()
                    )

            image_map[
                article_url
            ] = {

                "image_urls":
                image_urls,

                "image_captions":
                image_captions,

                "image_count":
                len(image_urls)

            }

    except Exception as e:

        print(e)


# =====================================
# ADD IMAGE DATA
# =====================================

image_urls_col = []

image_captions_col = []

image_count_col = []


for url in df["url"]:

    if url in image_map:

        image_urls_col.append(

            " | ".join(

                image_map[url][
                    "image_urls"
                ]

            )

        )

        image_captions_col.append(

            " | ".join(

                image_map[url][
                    "image_captions"
                ]

            )

        )

        image_count_col.append(

            image_map[url][
                "image_count"
            ]

        )

    else:

        image_urls_col.append("")
        image_captions_col.append("")
        image_count_col.append(0)


df["image_urls"] = (
    image_urls_col
)

df["image_captions"] = (
    image_captions_col
)

df["image_count"] = (
    image_count_col
)


# =====================================
# DIAGNOSTICS
# =====================================

print("\narticles with images\n")

print(

    (df["image_count"] > 0)
    .sum()

)


print("\nimage count distribution\n")

print(

    df["image_count"]
    .describe()

)


# =====================================
# SAVE
# =====================================

df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nfinal dataset saved\n")

print(output_file)


print("\nfinished")

In [ ]:
import pandas as pd

df = pd.read_csv(
    "eldiario_ukraine_russia_with_images.csv"
)

print()

print("corpus size")
print(len(df))

print()

print("articles with images")
print(
    (df["image_count"] > 0).sum()
)

print()

print("missing image urls")
print(
    df["image_urls"]
    .fillna("")
    .eq("")
    .sum()
)

print()

print("top image counts")
print(
    df["image_count"]
    .value_counts()
    .sort_index()
)

In [ ]:
import pandas as pd


df = pd.read_csv(
    "eldiario_ukraine_russia_with_images.csv"
)


all_images = []


for value in df["image_urls"].fillna(""):

    if not value.strip():
        continue

    urls = [

        x.strip()

        for x in value.split("|")

        if x.strip()

    ]

    all_images.extend(urls)


unique_images = set(
    all_images
)


print("\narticles\n")
print(len(df))

print("\nall image links\n")
print(len(all_images))

print("\nunique image links\n")
print(len(unique_images))

print("\nduplicate image links\n")
print(
    len(all_images)
    -
    len(unique_images)
)

In [ ]:
from collections import Counter


counter = Counter(
    all_images
)

print("\nmost reused images\n")

for url, n in counter.most_common(20):

    print(
        n,
        url
    )

In [ ]:
# This script creates two image metadata files from the filtered Ukraine–Russia corpus.
# First, it generates a complete article–image mapping table, preserving all links
# between articles and their associated images. Second, it extracts a deduplicated
# list of unique image URLs. This allows images to be downloaded only once while
# retaining the ability to trace each image back to all articles in which it appears.

In [ ]:
import pandas as pd


df = pd.read_csv(
    "eldiario_ukraine_russia_with_images.csv"
)

rows = []


for _, row in df.iterrows():

    article_url = row["url"]

    image_urls = str(
        row["image_urls"]
    )

    if image_urls == "nan":
        continue

    for img in image_urls.split("|"):

        img = img.strip()

        if not img:
            continue

        rows.append({

            "image_url": img,

            "article_url": article_url

        })


mapping = pd.DataFrame(
    rows
)

mapping.to_csv(

    "eldiario_image_article_mapping.csv",

    index=False,

    encoding="utf-8-sig"

)


unique_images = (

    mapping[
        ["image_url"]
    ]
    .drop_duplicates()

)

unique_images.to_csv(

    "eldiario_unique_images.csv",

    index=False,

    encoding="utf-8-sig"

)


print()

print(
    "article-image pairs:",
    len(mapping)
)

print()

print(
    "unique images:",
    len(unique_images)
)

In [ ]:
import pandas as pd
import requests
import hashlib
import os

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

from tqdm import tqdm


# =====================================
# SETTINGS
# =====================================

input_file = (
    "eldiario_unique_images.csv"
)

output_dir = (
    "eldiario_images"
)

failed_file = (
    "eldiario_failed_images.csv"
)

max_workers = 5


# =====================================
# CREATE FOLDER
# =====================================

os.makedirs(
    output_dir,
    exist_ok=True
)


# =====================================
# LOAD URLS
# =====================================

df = pd.read_csv(
    input_file
)

image_urls = (

    df["image_url"]
    .dropna()
    .astype(str)
    .unique()

)

print("\nunique images\n")
print(len(image_urls))


# =====================================
# SESSION
# =====================================

session = requests.Session()

session.headers.update({

    "User-Agent":
    "Mozilla/5.0"

})


# =====================================
# DOWNLOAD
# =====================================

def download_image(url):

    try:

        file_hash = hashlib.md5(

            url.encode(
                "utf-8"
            )

        ).hexdigest()

        ext = ".jpg"

        lower = url.lower()

        if ".png" in lower:
            ext = ".png"

        elif ".webp" in lower:
            ext = ".webp"

        elif ".jpeg" in lower:
            ext = ".jpeg"

        filename = (
            file_hash + ext
        )

        filepath = os.path.join(

            output_dir,

            filename

        )

        if os.path.exists(
            filepath
        ):

            return (
                url,
                "exists"
            )

        r = session.get(

            url,

            timeout=60,

            stream=True

        )

        if r.status_code != 200:

            return (

                url,

                f"status_{r.status_code}"

            )

        with open(

            filepath,

            "wb"

        ) as f:

            for chunk in r.iter_content(
                8192
            ):

                if chunk:

                    f.write(chunk)

        return (
            url,
            "success"
        )

    except Exception as e:

        return (
            url,
            str(e)
        )


# =====================================
# DOWNLOAD ALL
# =====================================

failed = []


with ThreadPoolExecutor(

    max_workers=max_workers

) as executor:

    futures = {

        executor.submit(
            download_image,
            url
        ): url

        for url in image_urls

    }

    for future in tqdm(

        as_completed(futures),

        total=len(futures),

        desc="downloading images"

    ):

        url, status = (
            future.result()
        )

        if status not in [

            "success",
            "exists"

        ]:

            failed.append({

                "image_url": url,

                "reason": status

            })


# =====================================
# SAVE FAILED
# =====================================

failed_df = pd.DataFrame(
    failed
)

failed_df.to_csv(

    failed_file,

    index=False,

    encoding="utf-8-sig"

)


# =====================================
# SUMMARY
# =====================================

print("\nfailed images\n")
print(len(failed_df))

print("\nimages folder\n")
print(output_dir)

print("\nfinished")

Нужны только 
eldiario_full_corpus_with_fulltext.csv
eldiario_ukraine_russia_with_images.csv
eldiario_image_article_mapping.csv
eldiario_images/